# Kota Architecture & The Event Stream

In this notebook, you'll learn how to:
1. Understand Kota's core asynchronous event loop built on **Tokio**
2. Simulate connecting to the **Axum** backend event channel
3. Process and visualize streaming tokens and tool calls in real-time
4. See how Kota separates backend reasoning from frontend UI rendering

## Setup: The Event Bus

Kota runs as a background daemon, broadcasting state changes (like LLM tokens streaming in or bash commands starting) over a typed `tokio::sync::broadcast::channel`.

Since we don't have the Rust binary running in this Jupyter environment, we will mock the stream using Python's `asyncio` to demonstrate exactly how frontend clients consume Kota's data.

In [3]:
import asyncio
import json
import sys


async def mock_kota_backend():
    """Simulates the Axum server yielding JSON events."""
    events = [
        {"type": "StepStarted", "step": 1, "tokens_in": 850},
        {"type": "Token", "text": "I need to "},
        {"type": "Token", "text": "check the "},
        {"type": "Token", "text": "directory.\n"},
        {
            "type": "ToolCallStarted",
            "step": 1,
            "tool": "run_command",
            "args": {"command": "ls -la"},
        },
        {
            "type": "ToolCallFinished",
            "step": 1,
            "tool": "run_command",
            "duration_ms": 45,
            "success": True,
        },
        {"type": "Done", "step": 1, "total_tokens": 920, "duration_ms": 1200},
    ]

    for event in events:
        yield json.dumps(event)
        await asyncio.sleep(0.3)  # Simulate network/LLM delay

## Subscribing to the Stream

When you use a TUI (Text User Interface) or a web dashboard to interact with Kota, it subscribes to the stream above. 
Let's write a simple client parser that intercepts `Token` events and prints them seamlessly, just like a real chatbot interface.

In [4]:
async def render_tui():
    print("╭─ Kota Mock TUI ─────────────────────────╮")

    async for raw_event in mock_kota_backend():
        event = json.loads(raw_event)

        if event["type"] == "StepStarted":
            print(
                f"\n│ ── Step {event['step']} ({event['tokens_in']} context tokens) ──"
            )
            sys.stdout.write("│ 💭 ")

        elif event["type"] == "Token":
            # Stream text directly to the console
            sys.stdout.write(event["text"])
            sys.stdout.flush()

        elif event["type"] == "ToolCallStarted":
            print(f"\n│ 🔧 Executing: {event['tool']}({event['args']})")

        elif event["type"] == "ToolCallFinished":
            status = "✓" if event["success"] else "❌"
            print(f"│ {status} Completed in {event['duration_ms']}ms")

        elif event["type"] == "Done":
            print(f"│ ── Done. Total Tokens: {event['total_tokens']} ──")

    print("╰─────────────────────────────────────────╯")


# Run the simulation
await render_tui()

╭─ Kota Mock TUI ─────────────────────────╮

│ ── Step 1 (850 context tokens) ──
│ 💭 I need to check the directory.

│ 🔧 Executing: run_command({'command': 'ls -la'})
│ ✓ Completed in 45ms
│ ── Done. Total Tokens: 920 ──
╰─────────────────────────────────────────╯


## Handling Tool Calls

Notice how the backend separates `Token` streams from `ToolCallStarted`. This design choice is critical for **Human-In-The-Loop (HITL)** safety. 
By capturing `ToolCallStarted` distinctly on the event bus, a UI client can pause execution and prompt the user to click `[Approve]` or `[Deny]` before the backend physically executes the bash command.

## Summary


In this notebook, you:

- Simulated Kota's asynchronous `tokio` event broadcaster.
- Rendered streaming LLM tokens smoothly alongside structural step updates.
- Learned how the strict event-typing architecture enables safe, interceptable Tool execution.